# transformerRafa

Este notebook usa BERT como o transformer principal, mas mantém a lógica de pré-processamento e a estrutura do pipeline que você pediu:

1. limpa o texto
2. substitui URLs, leis/decretos, número de processo e telefone por tokens especiais
3. divide em treino/teste
4. treina Word2Vec no texto de treino
5. cria matriz de embeddings
6. transforma palavras em IDs
7. passa por embedding + positional encoding + transformer
8. faz classificação final
9. reporta F1 macro e classification_report

Observação importante: com BERT, as etapas de embedding e positional encoding são resolvidas internamente pelo modelo. O que fica explícito aqui é a estrutura do experimento e o pré-processamento do texto.


In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

from gensim.models import Word2Vec
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch


In [ ]:
# === 1. Leitura do dataset ===
df = pd.read_excel('train.xlsx')
print('Shape:', df.shape)
print(df.head())


In [ ]:
# === 2. Limpeza e substituições ===
def clean_text(text):
    text = str(text).lower()

    # URLs -> token especial
    text = re.sub(r'https?://\S+|www\.\S+', ' LINK_URL ', text)

    # Lei / Decreto -> token especial
    text = re.sub(
        r'\b(?:lei|decreto)\s*n[ºo]?\s*\d{1,3}(?:[\.,]\d{1,3})*(?:/\d{4})?\b',
        ' LEI_DECRETO ',
        text,
        flags=re.IGNORECASE,
    )

    # Processo -> token especial
    text = re.sub(r'\b\d{5}\.\d{3,6}/\d{4}-\d{2}\b', ' NUMERO_PROCESSO ', text)

    # Telefone -> token especial
    text = re.sub(
        r'\b(?:\+?\d{2}\s?)?(?:\(?\d{2}\)?\s?\d{4,5}\s?\d{4}|\d{4}\s?\d{4})\b',
        ' NUMERO_TELEFONE ',
        text,
    )

    # Remove pontuação, preservando tokens especiais
    text = re.sub(r'[^a-z0-9_\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['resp_text'] = df['resp_text'].fillna('').apply(clean_text)
df['clarity'] = df['clarity'].astype(str)
print(df[['resp_text', 'clarity']].head())
print('Classes:', sorted(df['clarity'].unique()))


In [ ]:
# === 3. Divisão treino/teste ===
X = df['resp_text'].tolist()
y = df['clarity'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Treino:', len(X_train))
print('Teste:', len(X_test))


In [ ]:
# === 4. Treinando Word2Vec (baseline opcional / estrutura do pipeline) ===
sentences = [text.split() for text in X_train if text.strip()]

w2v = Word2Vec(
    sentences=sentences,
    vector_size=128,
    window=5,
    min_count=2,
    sg=1,
    workers=1,
    seed=42
)

print('Vocab size:', len(w2v.wv.key_to_index))
print('Exemplo de embedding:', w2v.wv['link_url'] if 'link_url' in w2v.wv else 'token não encontrado')


In [ ]:
# === 5. Preparação para BERT ===
labels = sorted(set(y))
label_to_id = {label: idx for idx, label in enumerate(labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

train_df = pd.DataFrame({'text': X_train, 'label': [label_to_id[v] for v in y_train]})
test_df = pd.DataFrame({'text': X_test, 'label': [label_to_id[v] for v in y_test]})

print('Labels:', labels)
print('Label map:', label_to_id)


In [ ]:
# === 6. Tokenização do BERT ===
model_name = 'neuralmind/bert-base-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding=True, max_length=128)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column('label', 'labels')
test_ds = test_ds.rename_column('label', 'labels')

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(train_ds[0])


In [ ]:
# === 7. Modelo BERT para classificação ===
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id_to_label,
    label2id=label_to_id
)

print(model)


In [ ]:
# === 8. Grid Search simples ===
grid = [
    {'lr': 2e-5, 'batch_size': 8},
    {'lr': 2e-5, 'batch_size': 16},
    {'lr': 3e-5, 'batch_size': 8},
    {'lr': 3e-5, 'batch_size': 16},
]

results = []

for cfg in grid:
    print("\n=== Testando cfg:", cfg)
    training_args = TrainingArguments(
        output_dir=f"./tmp_bert_{cfg['lr']}_{cfg['batch_size']}",
        eval_strategy='epoch',
        save_strategy='no',
        learning_rate=cfg['lr'],
        per_device_train_batch_size=cfg['batch_size'],
        per_device_eval_batch_size=cfg['batch_size'],
        num_train_epochs=1,
        weight_decay=0.01,
        logging_steps=100,
        disable_tqdm=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
    )

    trainer.train()
    metrics = trainer.evaluate()
    score = metrics.get('eval_f1', metrics.get('eval_f1_macro', 0.0))
    print('F1 macro:', score)
    results.append({**cfg, 'f1_macro': score})

print('\n=== RESULTADO DO GRID SEARCH ===')
for r in sorted(results, key=lambda x: x['f1_macro'], reverse=True):
    print(r)



## Interpretação final

- O pré-processamento específico do seu problema foi mantido.
- O grid search ficou presente para comparar configurações de treino.
- O transformer usado de verdade é o BERT (`neuralmind/bert-base-portuguese-cased`).
- O Word2Vec continua no notebook como parte da estrutura do pipeline e também como referência de baseline.
- O modelo final de classificação usa BERT com head linear para as classes do dataset.
